# Dịch Máy Tự Động English-French với Attention và Beam Search

**Đồ án:** Dịch máy từ tiếng Anh sang tiếng Pháp sử dụng LSTM + Bahdanau Attention + Beam Search

**Mục tiêu:** Xây dựng mô hình Seq2Seq với cơ chế Attention và so sánh Greedy vs Beam Search

## Mục lục
1. [Tải và xử lý dữ liệu](#1-load-wmt-2014-dataset)
2. [Kiến trúc mô hình với Attention](#2-model-2-layers-lstm--bahdanau-attention)
3. [Huấn luyện](#3-training)
4. [Beam Search](#4-beam-search-k4-trong-khoảng-3-5)
5. [So sánh BLEU Score](#5-bleu-score---so-sánh-greedy-vs-beam-search)

In [1]:
# Import các thư viện
import torch
import torch.nn as nn
import torch.optim as optim
import spacy
import numpy as np
import random
import os
from collections import Counter
import math
import time

SEED = 1234
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.backends.cudnn.deterministic = True

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Đang sử dụng thiết bị: {device}')

Đang sử dụng thiết bị: cpu


## 1. Load WMT 2014 Dataset

In [2]:
# Tokenizers
spacy_en = spacy.load('en_core_web_sm')
spacy_fr = spacy.load('fr_core_news_sm')

def tokenize_en(text):
    return [tok.text.lower() for tok in spacy_en.tokenizer(text)]

def tokenize_fr(text):
    return [tok.text.lower() for tok in spacy_fr.tokenizer(text)]

In [3]:
# Load data từ WMT 2014
def load_data(en_path, fr_path):
    """Đọc file dữ liệu song parallel English-French"""
    with open(en_path, 'r', encoding='utf-8') as f:
        en_sentences = [line.strip() for line in f]
    with open(fr_path, 'r', encoding='utf-8') as f:
        fr_sentences = [line.strip() for line in f]
    return list(zip(en_sentences, fr_sentences))

# Xác định thư mục dataset
dataset_path = 'dataset_wmt2014' if os.path.exists('dataset_wmt2014') else '../dataset_wmt2014'

train_data = load_data(f'{dataset_path}/train.en', f'{dataset_path}/train.fr')
val_data = load_data(f'{dataset_path}/val.en', f'{dataset_path}/val.fr')
test_data = load_data(f'{dataset_path}/test.en', f'{dataset_path}/test.fr')

print(f'Tập train: {len(train_data):,} cặp câu')
print(f'Tập validation: {len(val_data):,} cặp câu')
print(f'Tập test: {len(test_data):,} cặp câu')

Tập train: 165,380 cặp câu
Tập validation: 9,188 cặp câu
Tập test: 9,188 cặp câu


In [4]:
# Xây dựng vocabulary
def build_vocab(data, tokenizer, max_size=5000):
    counter = Counter()
    for src, trg in data:
        counter.update(tokenizer(src))
    most_common = counter.most_common(max_size)
    vocab = {'<pad>': 0, '<sos>': 1, '<eos>': 2, '<unk>': 3}
    for word, _ in most_common:
        vocab[word] = len(vocab)
    return vocab

en_vocab = build_vocab(train_data, tokenize_en)
fr_vocab = build_vocab(train_data, tokenize_fr)

print(f'EN vocab: {len(en_vocab):,}')
print(f'FR vocab: {len(fr_vocab):,}')

EN vocab: 5,004
FR vocab: 5,004


In [5]:
# Dataset & DataLoader
class TranslationDataset:
    def __init__(self, data, src_vocab, trg_vocab, src_tokenizer, trg_tokenizer):
        self.data = data
        self.src_vocab = src_vocab
        self.trg_vocab = trg_vocab
        self.src_tokenizer = src_tokenizer
        self.trg_tokenizer = trg_tokenizer
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        src, trg = self.data[idx]
        src_ids = [self.src_vocab.get(t, 3) for t in self.src_tokenizer(src)]
        trg_ids = [1] + [self.trg_vocab.get(t, 3) for t in self.trg_tokenizer(trg)] + [2]
        return torch.LongTensor(src_ids), torch.LongTensor(trg_ids)

def collate_fn(batch):
    src_batch, trg_batch = zip(*batch)
    max_src = max(len(s) for s in src_batch)
    max_trg = max(len(t) for t in trg_batch)
    src_padded = torch.zeros(len(batch), max_src, dtype=torch.long)
    trg_padded = torch.zeros(len(batch), max_trg, dtype=torch.long)
    for i, (src, trg) in enumerate(batch):
        src_padded[i, :len(src)] = src
        trg_padded[i, :len(trg)] = trg
    return src_padded, trg_padded

from torch.utils.data import DataLoader

BATCH_SIZE = 64
train_dataset = TranslationDataset(train_data, en_vocab, fr_vocab, tokenize_en, tokenize_fr)
val_dataset = TranslationDataset(val_data, en_vocab, fr_vocab, tokenize_en, tokenize_fr)
test_dataset = TranslationDataset(test_data, en_vocab, fr_vocab, tokenize_en, tokenize_fr)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, collate_fn=collate_fn)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, collate_fn=collate_fn)

## 2. Model: 2 Layers LSTM + Bahdanau Attention

In [6]:
# Encoder với 2 lớp Bidirectional LSTM
class Encoder(nn.Module):
    def __init__(self, input_dim, emb_dim, hid_dim, n_layers, dropout):
        super().__init__()
        self.embedding = nn.Embedding(input_dim, emb_dim, padding_idx=0)
        # Bidirectional LSTM với 2 layers
        self.rnn = nn.LSTM(emb_dim, hid_dim, n_layers, dropout=dropout, 
                          bidirectional=True, batch_first=True)
        # Project từ 2*hid_dim xuống hid_dim
        self.fc_hidden = nn.Linear(hid_dim * 2, hid_dim)
        self.fc_cell = nn.Linear(hid_dim * 2, hid_dim)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, src):
        embedded = self.dropout(self.embedding(src))
        outputs, (hidden, cell) = self.rnn(embedded)
        
        # Kết hợp forward và backward của lớp cuối
        hidden = torch.cat([hidden[-2], hidden[-1]], dim=1)
        hidden = torch.tanh(self.fc_hidden(hidden))
        cell = torch.cat([cell[-2], cell[-1]], dim=1)
        cell = torch.tanh(self.fc_cell(cell))
        
        # Repeat cho n_layers
        hidden = hidden.unsqueeze(0).repeat(self.rnn.num_layers, 1, 1)
        cell = cell.unsqueeze(0).repeat(self.rnn.num_layers, 1, 1)
        
        return outputs, hidden, cell

In [7]:
# Bahdanau Attention
class BahdanauAttention(nn.Module):
    def __init__(self, hid_dim):
        super().__init__()
        self.attn = nn.Linear(hid_dim * 3, hid_dim)
        self.v = nn.Linear(hid_dim, 1, bias=False)
        
    def forward(self, hidden, encoder_outputs):
        # hidden: [batch, hid_dim]
        # encoder_outputs: [batch, src_len, hid_dim*2]
        src_len = encoder_outputs.shape[1]
        hidden = hidden.unsqueeze(1).repeat(1, src_len, 1)
        # Tính attention energy
        energy = torch.tanh(self.attn(torch.cat([hidden, encoder_outputs], dim=2)))
        attention = self.v(energy).squeeze(2)
        return torch.softmax(attention, dim=1)

In [8]:
# Decoder với 2 lớp LSTM và Attention
class Decoder(nn.Module):
    def __init__(self, output_dim, emb_dim, hid_dim, n_layers, dropout, attention):
        super().__init__()
        self.output_dim = output_dim
        self.attention = attention
        self.embedding = nn.Embedding(output_dim, emb_dim, padding_idx=0)
        self.rnn = nn.LSTM(emb_dim + hid_dim * 2, hid_dim, n_layers, 
                          dropout=dropout, batch_first=True)
        self.fc_out = nn.Linear(hid_dim * 3 + emb_dim, output_dim)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, input, hidden, cell, encoder_outputs):
        input = input.unsqueeze(1)
        embedded = self.dropout(self.embedding(input))
        
        # Tính attention weights
        a = self.attention(hidden[-1], encoder_outputs)
        a = a.unsqueeze(1)
        
        # Context vector = weighted sum
        weighted = torch.bmm(a, encoder_outputs)
        rnn_input = torch.cat([embedded, weighted], dim=2)
        
        output, (hidden, cell) = self.rnn(rnn_input, (hidden, cell))
        
        # Dự đoán từ tiếp theo
        prediction = self.fc_out(torch.cat([
            output.squeeze(1), weighted.squeeze(1), embedded.squeeze(1)
        ], dim=1))
        
        return prediction, hidden, cell

In [9]:
# Seq2Seq Model
class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, device):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device = device
        
    def forward(self, src, trg, teacher_forcing_ratio=0.5):
        batch_size, trg_len = src.shape[0], trg.shape[1]
        trg_vocab_size = self.decoder.output_dim
        
        outputs = torch.zeros(batch_size, trg_len, trg_vocab_size).to(self.device)
        encoder_outputs, hidden, cell = self.encoder(src)
        
        input = trg[:, 0]
        for t in range(1, trg_len):
            output, hidden, cell = self.decoder(input, hidden, cell, encoder_outputs)
            outputs[:, t] = output
            teacher_force = random.random() < teacher_forcing_ratio
            top1 = output.argmax(1)
            input = trg[:, t] if teacher_force else top1
        
        return outputs

In [10]:
# Khởi tạo model với 2 layers
INPUT_DIM = len(en_vocab)
OUTPUT_DIM = len(fr_vocab)
EMB_DIM = 256
HID_DIM = 256
N_LAYERS = 2  # YÊU CẦU: 2 lớp LSTM
DROPOUT = 0.5

attention = BahdanauAttention(HID_DIM)
enc = Encoder(INPUT_DIM, EMB_DIM, HID_DIM, N_LAYERS, DROPOUT)
dec = Decoder(OUTPUT_DIM, EMB_DIM, HID_DIM, N_LAYERS, DROPOUT, attention)
model = Seq2Seq(enc, dec, device).to(device)

print(f'Tổng parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')

Tổng parameters: 12,357,516


## 3. Training

In [ ]:
optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss(ignore_index=0)

def train_epoch(model, loader, optimizer, criterion, clip, print_every=100):
    model.train()
    epoch_loss = 0
    start_time = time.time()
    
    for i, (src, trg) in enumerate(loader):
        src, trg = src.to(device), trg.to(device)
        optimizer.zero_grad()
        
        output = model(src, trg)
        output_dim = output.shape[-1]
        
        output = output[:, 1:].reshape(-1, output_dim)
        trg = trg[:, 1:].reshape(-1)
        
        loss = criterion(output, trg)
        loss.backward()
        
        torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
        optimizer.step()
        
        epoch_loss += loss.item()
        if (i + 1) % print_every == 0:
            avg_loss = epoch_loss / (i + 1)
            elapsed = time.time() - start_time
            print(f'Batch {i+1}/{len(loader)} | Time: {elapsed:.0f}s | Loss: {avg_loss:.4f}')
            
    return epoch_loss / len(loader)

def evaluate(model, loader, criterion):
    model.eval()
    epoch_loss = 0
    with torch.no_grad():
        for src, trg in loader:
            src, trg = src.to(device), trg.to(device)
            output = model(src, trg, 0)
            output_dim = output.shape[-1]
            output = output[:, 1:].reshape(-1, output_dim)
            trg = trg[:, 1:].reshape(-1)
            loss = criterion(output, trg)
            epoch_loss += loss.item()
    return epoch_loss / len(loader)

: 

In [ ]:
# Training loop
N_EPOCHS = 10
CLIP = 1
PATIENCE = 3

best_val_loss = float('inf')
patience_counter = 0

print('Bắt đầu huấn luyện...\n')

for epoch in range(N_EPOCHS):
    start_time = time.time()
    
    train_loss = train_epoch(model, train_loader, optimizer, criterion, CLIP)
    val_loss = evaluate(model, val_loader, criterion)
    
    end_time = time.time()
    elapsed_mins = int((end_time - start_time) / 60)
    elapsed_secs = int((end_time - start_time) - (elapsed_mins * 60))
    
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        torch.save(model.state_dict(), 'best_model_attention.pth')
        print(f'Epoch: {epoch+1:02} | Thời gian: {elapsed_mins}m {elapsed_secs}s')
        print(f'\tTrain Loss: {train_loss:.3f} | Train PPL: {math.exp(train_loss):7.3f}')
        print(f'\t Val. Loss: {val_loss:.3f} |  Val. PPL: {math.exp(val_loss):7.3f}')
        print(f'\t*** Đã lưu mô hình tốt nhất! ***\n')
    else:
        patience_counter += 1
        print(f'Epoch: {epoch+1:02} | Thời gian: {elapsed_mins}m {elapsed_secs}s')
        print(f'\tTrain Loss: {train_loss:.3f} | Train PPL: {math.exp(train_loss):7.3f}')
        print(f'\t Val. Loss: {val_loss:.3f} |  Val. PPL: {math.exp(val_loss):7.3f}')
        print(f'\tPatience: {patience_counter}/{PATIENCE}\n')
        
        if patience_counter >= PATIENCE:
            print(f'Dừng sớm sau {epoch+1} epochs')
            break

print('Hoàn thành huấn luyện!\n')

# Tải và đánh giá mô hình tốt nhất
model.load_state_dict(torch.load('best_model_attention.pth'))
print('Đã tải mô hình tốt nhất!')

test_loss = evaluate(model, test_loader, criterion)
print(f'\nTest Loss: {test_loss:.3f} | Test PPL: {math.exp(test_loss):7.3f}')


Bắt đầu huấn luyện...

Batch 100/2585 | Time: 1396s | Loss: 2.0537
Batch 200/2585 | Time: 2754s | Loss: 1.8569
Batch 300/2585 | Time: 3877s | Loss: 1.7860
Batch 400/2585 | Time: 5489s | Loss: 1.7452
Batch 500/2585 | Time: 7393s | Loss: 1.7206
Batch 600/2585 | Time: 8735s | Loss: 1.7031


## 4. Beam Search (k=4, trong khoảng 3-5)

In [ ]:
def beam_search_decode(model, src, beam_width, max_len=50):
    """Beam Search với beam_width=4 (trong khoảng 3-5)"""
    model.eval()
    
    with torch.no_grad():
        encoder_outputs, hidden, cell = model.encoder(src)
        
        # Khởi tạo beam với <sos>
        beams = [([1], 0.0, hidden, cell)]
        
        for _ in range(max_len):
            candidates = []
            
            for seq, score, h, c in beams:
                if seq[-1] == 2:  # <eos>
                    candidates.append((seq, score, h, c))
                    continue
                
                input_token = torch.LongTensor([seq[-1]]).to(device)
                output, h_new, c_new = model.decoder(input_token, h, c, encoder_outputs)
                log_probs = torch.log_softmax(output, dim=1)
                
                top_k = torch.topk(log_probs, beam_width)
                
                for i in range(beam_width):
                    token = top_k.indices[0][i].item()
                    token_score = top_k.values[0][i].item()
                    candidates.append((seq + [token], score + token_score, h_new, c_new))
            
            # Chọn top beam_width candidates
            beams = sorted(candidates, key=lambda x: x[1], reverse=True)[:beam_width]
            
            if all(seq[-1] == 2 for seq, _, _, _ in beams):
                break
        
        return beams[0][0][1:]  # Bỏ <sos>

def greedy_decode(model, src, max_len=50):
    """Greedy decoding (beam_width=1)"""
    return beam_search_decode(model, src, beam_width=1, max_len=max_len)

## 5. BLEU Score - So sánh Greedy vs Beam Search

In [ ]:
def compute_bleu(reference, candidate, max_n=4):
    """Tính BLEU score"""
    ref_tokens = reference.split()
    cand_tokens = candidate.split()
    
    if len(cand_tokens) == 0:
        return 0.0
    
    # Brevity penalty
    bp = 1.0 if len(cand_tokens) > len(ref_tokens) else math.exp(1 - len(ref_tokens) / len(cand_tokens))
    
    # n-gram precisions
    precisions = []
    for n in range(1, max_n + 1):
        ref_ngrams = Counter([tuple(ref_tokens[i:i+n]) for i in range(len(ref_tokens) - n + 1)])
        cand_ngrams = Counter([tuple(cand_tokens[i:i+n]) for i in range(len(cand_tokens) - n + 1)])
        overlap = sum((ref_ngrams & cand_ngrams).values())
        total = sum(cand_ngrams.values())
        precisions.append(overlap / total if total > 0 else 0.0)
    
    if min(precisions) > 0:
        geo_mean = math.exp(sum(math.log(p) for p in precisions) / len(precisions))
        return bp * geo_mean
    return 0.0

def tokens_to_sentence(tokens, vocab):
    inv_vocab = {v: k for k, v in vocab.items()}
    words = []
    for t in tokens:
        if t == 2:  # <eos>
            break
        if t not in [0, 1, 2, 3]:
            words.append(inv_vocab.get(t, '<unk>'))
    return ' '.join(words)

In [ ]:
# So sánh trên 100 câu test
model.eval()
test_samples = test_data[:100]

bleu_greedy = []
bleu_beam = []

BEAM_SIZE = 4  # Chọn k=4 (trong khoảng 3-5)

print(f'Đang đánh giá với Beam Size = {BEAM_SIZE}...')
for i, (src_text, trg_text) in enumerate(test_samples):
    if i % 20 == 0:
        print(f'  {i}/{len(test_samples)}')
    
    src_tokens = tokenize_en(src_text)
    src_ids = torch.LongTensor([[en_vocab.get(t, 3) for t in src_tokens]]).to(device)
    
    reference = ' '.join(tokenize_fr(trg_text))
    
    # Greedy
    greedy_ids = greedy_decode(model, src_ids)
    greedy_text = tokens_to_sentence(greedy_ids, fr_vocab)
    bleu_greedy.append(compute_bleu(reference, greedy_text))
    
    # Beam Search k=4
    beam_ids = beam_search_decode(model, src_ids, beam_width=BEAM_SIZE)
    beam_text = tokens_to_sentence(beam_ids, fr_vocab)
    bleu_beam.append(compute_bleu(reference, beam_text))

print('\n' + '='*60)
print('KẾT QUẢ SO SÁNH BLEU SCORE')
print('='*60)
print(f'Greedy (k=1):        {np.mean(bleu_greedy):.4f}')
print(f'Beam Search (k={BEAM_SIZE}):    {np.mean(bleu_beam):.4f}')
print(f'Cải thiện:           +{np.mean(bleu_beam)-np.mean(bleu_greedy):.4f}')
print('='*60)

In [ ]:
# Ví dụ dịch
print('\nVÍ DỤ DỊCH:')
print('='*60)

for i in range(3):
    src_text, trg_text = test_samples[i]
    src_tokens = tokenize_en(src_text)
    src_ids = torch.LongTensor([[en_vocab.get(t, 3) for t in src_tokens]]).to(device)
    
    greedy_ids = greedy_decode(model, src_ids)
    beam_ids = beam_search_decode(model, src_ids, beam_width=BEAM_SIZE)
    
    print(f'\n[{i+1}] EN: {src_text}')
    print(f'    Tham chiếu:  {trg_text}')
    print(f'    Greedy:      {tokens_to_sentence(greedy_ids, fr_vocab)}')
    print(f'    Beam (k={BEAM_SIZE}):  {tokens_to_sentence(beam_ids, fr_vocab)}')